In [ ]:
import cv2
from pathlib import Path

RAW_DIR      = "Gesture Image Data" #image directory
GRAY_DIR     = "data/grayscale"
IMG_SIZE     = 50

total = 0

for class_folder in sorted(Path(RAW_DIR).iterdir()):
    if not class_folder.is_dir():
        continue

    gray_class = Path(GRAY_DIR) / class_folder.name
    gray_class.mkdir(parents=True, exist_ok=True)

    images = sorted(class_folder.glob("*.jpg"))

    for img_path in images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(str(gray_class / img_path.name), resized)
        total += 1

    print(f"✓ {class_folder.name}  —  {len(images)} images")

print(f"\nDone! {total} grayscale images saved to '{GRAY_DIR}'")

✓ 0  —  1500 images
✓ 1  —  1500 images
✓ 2  —  1500 images
✓ 3  —  1500 images
✓ 4  —  1500 images
✓ 5  —  1500 images
✓ 6  —  1500 images
✓ 7  —  1500 images
✓ 8  —  1500 images
✓ 9  —  1500 images
✓ A  —  1500 images
✓ B  —  1500 images
✓ C  —  1500 images
✓ D  —  1500 images
✓ E  —  1500 images
✓ F  —  1500 images
✓ G  —  1500 images
✓ H  —  1500 images
✓ I  —  1500 images
✓ J  —  1500 images
✓ K  —  1500 images
✓ L  —  1500 images
✓ M  —  1500 images
✓ N  —  1500 images
✓ Next  —  1500 images
✓ O  —  1500 images
✓ P  —  1500 images
✓ Q  —  1500 images
✓ R  —  1500 images
✓ S  —  1500 images
✓ T  —  1500 images
✓ U  —  1500 images
✓ V  —  1500 images
✓ W  —  1500 images
✓ X  —  1500 images
✓ Y  —  1500 images
✓ Z  —  1500 images

Done! 55500 grayscale images saved to 'data/grayscale'


In [7]:
#pickle 


import cv2
import pickle
import numpy as np
from pathlib import Path

GRAY_DIR    = "data/grayscale"
PICKLE_FILE = "data_pixels.pickle"
IMG_SIZE    = 50

data    = []
labels  = []
total   = 0

for class_folder in sorted(Path(GRAY_DIR).iterdir()):
    if not class_folder.is_dir():
        continue

    label       = class_folder.name
    images      = sorted(class_folder.glob("*.jpg"))
    class_count = 0

    for img_path in images:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        resized  = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        features = resized.flatten() / 255.0

        data.append(features)
        labels.append(label)
        class_count += 1
        total += 1

    print(f"✓ {label:>6}  —  {class_count} samples  |  total: {total}")

with open(PICKLE_FILE, "wb") as f:
    pickle.dump({"data": np.array(data), "labels": np.array(labels)}, f)

print(f"\nDone!")
print(f"  Pickle → {PICKLE_FILE}")
print(f"  Total samples : {len(data)}")
print(f"  Shape         : {np.array(data).shape}")

✓      0  —  1500 samples  |  total: 1500
✓      1  —  1500 samples  |  total: 3000
✓      2  —  1500 samples  |  total: 4500
✓      3  —  1500 samples  |  total: 6000
✓      4  —  1500 samples  |  total: 7500
✓      5  —  1500 samples  |  total: 9000
✓      6  —  1500 samples  |  total: 10500
✓      7  —  1500 samples  |  total: 12000
✓      8  —  1500 samples  |  total: 13500
✓      9  —  1500 samples  |  total: 15000
✓      A  —  1500 samples  |  total: 16500
✓      B  —  1500 samples  |  total: 18000
✓      C  —  1500 samples  |  total: 19500
✓      D  —  1500 samples  |  total: 21000
✓      E  —  1500 samples  |  total: 22500
✓      F  —  1500 samples  |  total: 24000
✓      G  —  1500 samples  |  total: 25500
✓      H  —  1500 samples  |  total: 27000
✓      I  —  1500 samples  |  total: 28500
✓      J  —  1500 samples  |  total: 30000
✓      K  —  1500 samples  |  total: 31500
✓      L  —  1500 samples  |  total: 33000
✓      M  —  1500 samples  |  total: 34500
✓      N  —  1500

In [8]:
#train

import pickle
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

data_dict = pickle.load(open("data_pixels.pickle", "rb"))
data      = data_dict["data"]
labels    = data_dict["labels"]

print(f"Dataset: {data.shape[0]} samples, {data.shape[1]} features, {len(set(labels))} classes")

x_train, x_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, shuffle=True, stratify=labels, random_state=42
)

print("Training Random Forest...")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
score  = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {score * 100:.2f}%")
print("\nPer-class report:")
print(classification_report(y_test, y_pred))

with open("model_rf.pkl", "wb") as f:
    pickle.dump({"model": model}, f)

print("Model saved → models/model_rf.pkl")

Dataset: 55500 samples, 2500 features, 37 classes
Training Random Forest...

Accuracy: 100.00%

Per-class report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       300
           1       1.00      1.00      1.00       300
           2       1.00      1.00      1.00       300
           3       1.00      1.00      1.00       300
           4       1.00      1.00      1.00       300
           5       1.00      1.00      1.00       300
           6       1.00      1.00      1.00       300
           7       1.00      1.00      1.00       300
           8       1.00      1.00      1.00       300
           9       1.00      1.00      1.00       300
           A       1.00      1.00      1.00       300
           B       1.00      1.00      1.00       300
           C       1.00      1.00      1.00       300
           D       1.00      1.00      1.00       300
           E       1.00      1.00      1.00       300
           F       1.

In [6]:
import cv2
import pickle
import numpy as np
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk

MODEL_PATH = "model_rf.pkl"
IMG_SIZE   = 50

model = pickle.load(open(MODEL_PATH, "rb"))["model"]

correct_total = 0
tested_total  = 0

root = tk.Tk()
root.title("RF Tester")
root.geometry("400x500")

btn = tk.Button(root, text="Browse Image", font=("Helvetica", 14),
                bg="green", fg="white", padx=20, pady=10)
btn.pack(pady=20)

img_label    = tk.Label(root)
img_label.pack()

result_label = tk.Label(root, text="", font=("Helvetica", 13))
result_label.pack(pady=10)

def browse():
    global correct_total, tested_total

    path = filedialog.askopenfilename(
        initialdir="data/grayscale",
        filetypes=[("JPEG", "*.jpg"), ("All", "*.*")]
    )
    if not path:
        return

    true_label = path.split("/")[-2]
    img        = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    resized    = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    features   = resized.flatten() / 255.0

    prediction = model.predict([features])[0]
    confidence = model.predict_proba([features]).max() * 100

    correct = prediction == true_label
    if correct:
        correct_total += 1
    tested_total += 1

    color = "green" if correct else "red"
    result_label.config(
        text=f"True: {true_label}\nPred: {prediction}  ({confidence:.1f}%)\n{'✓ CORRECT' if correct else '✗ WRONG'}",
        fg=color
    )

    # ── Terminal output ──────────────────────────────
    print("─" * 40)
    print(f"  Image      : {path.split('/')[-1]}")
    print(f"  True Label : {true_label}")
    print(f"  Predicted  : {prediction}")
    print(f"  Confidence : {confidence:.1f}%")
    print(f"  Result     : {'✓ CORRECT' if correct else '✗ WRONG'}")
    print(f"  Session    : {correct_total}/{tested_total} correct ({correct_total/tested_total*100:.1f}%)")
    print("─" * 40)

    pil_img = Image.fromarray(img).resize((250, 250))
    tk_img  = ImageTk.PhotoImage(pil_img)
    img_label.config(image=tk_img)
    img_label.image = tk_img

btn.config(command=browse)
root.mainloop()

print(f"\n{'='*40}")
print(f"  FINAL SESSION RESULTS")
print(f"  Tested  : {tested_total} images")
print(f"  Correct : {correct_total}")
print(f"  Accuracy: {correct_total/tested_total*100:.1f}%" if tested_total > 0 else "  No images tested")
print(f"{'='*40}")

────────────────────────────────────────
  Image      : 20.jpg
  True Label : Next
  Predicted  : Next
  Confidence : 100.0%
  Result     : ✓ CORRECT
  Session    : 1/1 correct (100.0%)
────────────────────────────────────────

  FINAL SESSION RESULTS
  Tested  : 1 images
  Correct : 1
  Accuracy: 100.0%
